# 📝 [제 3편] 벡터와 행렬 - numpy 로 데이터 다루기

**학년/반/번호:** 2학년 ( )반 ( )번  **이름:** (           )

> 근거 자료: 박성호, 「Python numpy (I)·(II)·(III)」 (ML_LEC_06, 07, 08)

---

## 🎯 학습 목표
1. numpy 로 벡터와 행렬을 만들고 형상(shape)과 차원(ndim)을 확인할 수 있다.
2. 행렬 곱의 계산 원리와 크기 조건 (m × n)(n × k) = (m × k) 를 설명할 수 있다.
3. 브로드캐스트, 전치행렬, 인덱싱·슬라이싱으로 데이터를 원하는 모양으로 바꿀 수 있다.

---

## 🧭 지난 시간 복습
* 제 2편에서 여러 변수를 `np.array([1.0, 2.0])`, `np.array([[1.0, 2.0], [3.0, 4.0]])` 처럼 **배열**에 담아 수치미분했습니다.
* 오늘은 이 배열, 즉 **벡터와 행렬**을 자세히 배웁니다.

> **왜 행렬일까?** 🤔  
> 사진 한 장은 숫자(밝기)가 가로세로로 늘어선 **행렬**입니다. 28 × 28 손글씨 숫자 사진은 784 개의 숫자이고,  
> 인공지능은 이 숫자들에 **가중치 행렬**을 곱해서 "이 숫자는 5" 라고 판단합니다.

[⬅ 이전 : 제 2편 수치미분 - 파이썬으로 미분하기](https://colab.research.google.com/github/chenyenhee/ai-math-colab/blob/python/02_numerical_derivative.ipynb)　|　[📚 목차](https://colab.research.google.com/github/chenyenhee/ai-math-colab/blob/python/00_index.ipynb)　|　[다음 : 제 4편 머신러닝의 개념과 종류 ➡](https://colab.research.google.com/github/chenyenhee/ai-math-colab/blob/python/04_ml_overview.ipynb)

> 💡 **처음 여는 경우** : 파일 → **Drive에 사본 저장** 을 누르고, 내 사본에서 공부하세요. (빈칸을 채우고 저장할 수 있습니다)
>
> 🧹 **다 본 뒤에는** : 수정 → **모든 출력 지우기** 후 저장하면 애니메이션 때문에 무거워진 사본이 다시 가벼워집니다.

## ⚙️ 0단계. 준비하기
아래 셀을 **가장 먼저 한 번** 실행하세요. (그래프 한글 글꼴과 애니메이션 설정)

In [ ]:
#@title ▶ 준비 셀 (먼저 실행하세요)
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib import animation
from IPython.display import HTML, display
import urllib.request, os

font_path = 'NanumGothic.ttf'
try:
    if not os.path.exists(font_path):
        urllib.request.urlretrieve('https://raw.githubusercontent.com/google/fonts/main/ofl/nanumgothic/NanumGothic-Regular.ttf', font_path)
    fm.fontManager.addfont(font_path)
    plt.rc('font', family=fm.FontProperties(fname=font_path).get_name())
except Exception as e:
    print('글꼴을 내려받지 못했습니다(그래프의 한글이 깨질 수 있음):', e)
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['animation.embed_limit'] = 50
def show_anim(anim):
    # 애니메이션을 코랩 화면에 재생 버튼과 함께 보여 주기
    display(HTML(anim.to_jshtml()))
np.set_printoptions(precision=6, suppress=True)
print('✅ 준비 완료!')

## 📖 개념 다지기 1. 벡터와 행렬, 형상(shape)

* **벡터**: 숫자를 한 줄로 늘어놓은 것 → 1차원 배열.  예) A = (1, 2, 3)
* **행렬**: 숫자를 가로(행)·세로(열)로 늘어놓은 것 → 2차원 배열.

　　A = ( 1　2　3 )
　　　　( 4　5　6 )　　→ 행 2개, 열 3개 → **형상(shape) 2 × 3**

| 확인할 것 | numpy 코드 | 위 행렬 A 의 결과 |
|:---|:---|:---|
| 형상 | `A.shape` | (2, 3) |
| 차원 | `A.ndim` | 2 |
| 원소 개수 | `A.size` | 6 |

> **[연습 1]** 행렬 B = ( 1　2 ) / ( 3　4 ) / ( 5　6 ) (3행 2열) 의 형상은 ( 　 × 　 ), 원소 개수는 ( 　 ) 개

In [ ]:
A = np.array([1, 2, 3])                       # 벡터
B = np.array([[1, 2, 3], [4, 5, 6]])          # 행렬
print('A =', A, '  shape =', A.shape, '  ndim =', A.ndim)
print('B =\n', B)
print('B.shape =', B.shape, '  B.ndim =', B.ndim, '  B.size =', B.size)

## 📖 개념 다지기 2. 같은 자리끼리 하는 사칙연산과 형 변환(reshape)

크기가 같은 두 벡터(행렬)의 +, −, ×, ÷ 는 **같은 자리의 원소끼리** 계산합니다.

　A = (1, 2, 3),　B = (4, 5, 6) 이면
　A + B = (5, 7, 9),　A × B = ( 　, 　, 　 ),　B / A = (4, 2.5, 2)

**reshape** 는 원소의 순서는 그대로 두고 **모양만** 바꿉니다.
　(1, 2, 3, 4, 5, 6) → reshape(2, 3) → ( 1 2 3 / 4 5 6 )

In [ ]:
A = np.array([1, 2, 3])
B = np.array([4, 5, 6])
print('A + B =', A + B)
print('A - B =', A - B)
print('A * B =', A * B)
print('B / A =', B / A)

C = np.array([1, 2, 3, 4, 5, 6])
print('\nreshape(2, 3) =\n', C.reshape(2, 3))
print('reshape(3, 2) =\n', C.reshape(3, 2))

> **[확인 1]** `C.reshape(3, 2)` 의 결과를 손으로 먼저 써 보고 비교하세요. 원소는 ( 행 / 열 ) 방향으로 차례대로 채워집니다.
> **[생각해 보기]** `C.reshape(4, 2)` 는 왜 오류가 날까요? (                                       )

## 📖 개념 다지기 3. 행렬 곱 (dot product) ⭐ 가장 중요!

행렬 곱 A · B 의 (i 행, j 열) 원소 = **A 의 i 번째 행** 과 **B 의 j 번째 열** 에서 같은 순서의 원소끼리 곱해서 모두 더한 값

　A = ( 1　2　3 )　　B = ( −1　−2 )
　　　( 4　5　6 )　　　　( −3　−4 )
　　　　　　　　　　　　( −5　−6 )

* (1, 1) 원소 = 1 × (−1) + 2 × (−3) + 3 × (−5) = −1 − 6 − 15 = −22
* (1, 2) 원소 = 1 × (−2) + 2 × (−4) + 3 × (−6) = ( 　　 )
* (2, 1) 원소 = 4 × (−1) + 5 × (−3) + 6 × (−5) = ( 　　 )
* (2, 2) 원소 = ( 　　　　　　　　　　　　　　 ) = −64

### 크기 조건
**(m × n) · (n × k) = (m × k)**  → 앞 행렬의 **열의 개수** 와 뒤 행렬의 **행의 개수** 가 같아야 곱할 수 있습니다.

　(2 × <u>3</u>) · (<u>3</u> × 2) = (2 × 2)

## 🎬 애니메이션 1. 행렬 곱이 계산되는 순서

A 의 행(빨강)과 B 의 열(파랑)이 만나 결과 행렬 C 의 한 칸(초록)이 채워집니다. 실행 후 ▶ 를 누르세요.

In [ ]:
#@title 🎬 애니메이션 1 실행 (행렬 곱)
A = np.array([[1, 2, 3], [4, 5, 6]])
B = np.array([[-1, -2], [-3, -4], [-5, -6]])
C = A @ B
m, n = A.shape; k = B.shape[1]

def draw_matrix(ax, M, x0, y0, name, hl_rows=(), hl_cols=(), color='white', show=None):
    r, c = M.shape
    for i in range(r):
        for j in range(c):
            fc = 'white'
            if i in hl_rows: fc = '#ffb3b3'
            if j in hl_cols: fc = '#b3d1ff'
            if show is not None and (i, j) == show[0]: fc = show[1]
            ax.add_patch(plt.Rectangle((x0 + j, y0 - i), 1, 1, fc=fc, ec='k', lw=1.5))
            txt = '' if (show is not None and show[2] is not None and (i, j) not in show[2]) else str(M[i, j])
            ax.text(x0 + j + 0.5, y0 - i + 0.5, txt, ha='center', va='center', fontsize=15)
    ax.text(x0 + c/2, y0 + 1.4, f'{name}  ({r} × {c})', ha='center', fontsize=14, weight='bold')

steps = [(i, j) for i in range(m) for j in range(k)]
frames = []
for t, (i, j) in enumerate(steps):
    frames += [(t, i, j)] * 6
frames += [(len(steps), None, None)] * 8

fig, ax = plt.subplots(figsize=(11, 5))
def draw(f):
    t, i, j = frames[f]
    ax.clear(); ax.set_xlim(-0.5, 13); ax.set_ylim(-3.8, 3); ax.axis('off'); ax.set_aspect('equal')
    done = set(steps[:t])
    if i is None:
        draw_matrix(ax, A, 0, 1, 'A'); draw_matrix(ax, B, 4.5, 1.5, 'B')
        draw_matrix(ax, C, 9, 1, 'C = A · B', show=(None, None, None))
        ax.text(6.5, -3.3, '완성!  (2 × 3) · (3 × 2) = (2 × 2)', ha='center', fontsize=15, color='green', weight='bold')
        return
    draw_matrix(ax, A, 0, 1, 'A', hl_rows=(i,))
    draw_matrix(ax, B, 4.5, 1.5, 'B', hl_cols=(j,))
    draw_matrix(ax, C, 9, 1, 'C = A · B', show=((i, j), '#b3ffb3', done | {(i, j)}))
    terms = ' + '.join(f'{A[i, q]}×({B[q, j]})' for q in range(n))
    ax.text(6.5, -3.3, f'C[{i+1},{j+1}] = {terms} = {C[i, j]}', ha='center', fontsize=14, color='darkgreen')
    ax.text(3.6, 0.5, '·', fontsize=30, ha='center', va='center'); ax.text(8.3, 0.5, '=', fontsize=24, ha='center', va='center')

anim = animation.FuncAnimation(fig, draw, frames=len(frames), interval=200)
plt.close(fig)
show_anim(anim)

In [ ]:
A = np.array([[1, 2, 3], [4, 5, 6]])               # 2 × 3
B = np.array([[-1, -2], [-3, -4], [-5, -6]])       # 3 × 2
C = np.dot(A, B)                                   # A @ B 라고 써도 같음
print('A.shape =', A.shape, ', B.shape =', B.shape, '→ C.shape =', C.shape)
print(C)

> **[연습 2]** 크기 조건만 보고 결과의 형상을 쓰세요. 곱할 수 없으면 × 표시.
> ① (3 × 4) · (4 × 1) = ( 　　 ) 　 ② (2 × 2) · (3 × 2) = ( 　　 ) 　 ③ (1 × 784) · (784 × 100) = ( 　　 )

> 💡 **머신러닝과의 연결**: ③ 은 손글씨 사진 한 장(784개 숫자)에 가중치 행렬(784 × 100)을 곱하는 계산입니다.
> 행렬 곱 덕분에 (64 × 64) · (64 × 256) · (256 × 512) · … 처럼 **크기 조건만 맞으면 원하는 만큼 층을 이어 붙일 수 있습니다.** 이것이 딥러닝의 '깊은 층' 입니다.

⚠️ `A * B` (같은 자리끼리 곱) 와 `np.dot(A, B)` (행렬 곱) 는 **완전히 다릅니다!**

## 📖 개념 다지기 4. 브로드캐스트 (broadcast)

수학에서는 크기가 다른 행렬끼리 더할 수 없지만, numpy 는 **작은 쪽을 반복해서 크기를 맞춘 뒤** 계산해 줍니다.

　A = ( 1　2 ) / ( 3　4 ),　b = 5 일 때　A + b = ( 1　2 ) / ( 3　4 ) + ( 5　5 ) / ( 5　5 ) = ( 6　7 ) / ( 8　9 )

　C = ( 1　2 ) / ( 3　4 ),　D = ( 4　5 ) 일 때　C + D = ( 1　2 ) / ( 3　4 ) + ( 4　5 ) / ( 4　5 ) = ( 　　 ) / ( 　　 )

In [ ]:
#@title 🎬 애니메이션 2 실행 (브로드캐스트)
C = np.array([[1, 2], [3, 4]]); D = np.array([4, 5])
fig, ax = plt.subplots(figsize=(10, 3.8))
def cell(x, y, v, fc='white', alpha=1):
    ax.add_patch(plt.Rectangle((x, y), 1, 1, fc=fc, ec='k', lw=1.5, alpha=alpha))
    ax.text(x + .5, y + .5, str(v), ha='center', va='center', fontsize=16, alpha=alpha)
frames = list(range(0, 21)) + [20] * 10
def draw(f):
    ax.clear(); ax.set_xlim(-0.5, 10.5); ax.set_ylim(-1.2, 2.6); ax.axis('off'); ax.set_aspect('equal')
    for i in range(2):
        for j in range(2): cell(j, 1 - i, C[i, j])
    ax.text(2.5, 1, '+', fontsize=24, ha='center', va='center')
    for j in range(2): cell(3 + j, 1, D[j], '#b3d1ff')
    a = min(f / 10, 1)                               # 복사된 행이 내려오는 정도
    for j in range(2): cell(3 + j, 1 - a, D[j], '#ffe0a3', alpha=0.4 + 0.6*a)
    ax.text(3.9, 2.25, '작은 쪽(D)을 행 방향으로 복사', fontsize=12, ha='center', color='royalblue')
    if f >= 12:
        ax.text(5.5, 1, '=', fontsize=24, ha='center', va='center')
        R = C + D
        for i in range(2):
            for j in range(2): cell(6 + j, 1 - i, R[i, j], '#b3ffb3')
anim = animation.FuncAnimation(fig, draw, frames=frames, interval=120)
plt.close(fig)
show_anim(anim)

In [ ]:
A = np.array([[1, 2], [3, 4]])
print('A + 5 =\n', A + 5)
C = np.array([[1, 2], [3, 4]]); D = np.array([4, 5])
print('C + D =\n', C + D)

## 📖 개념 다지기 5. 전치행렬 (transpose)

행렬 A 의 **행과 열을 바꾼** 행렬을 전치행렬이라 하고 A<sup>T</sup> 로 씁니다. (1행 → 1열, 2행 → 2열, …)

　A = ( 1　2 ) / ( 3　4 ) / ( 5　6 )  (3 × 2)　→　A<sup>T</sup> = ( 1　3　5 ) / ( 2　4　6 )  ( 　 × 　 )

> 💡 행렬 곱의 크기가 안 맞을 때 전치행렬로 맞춰 줍니다.
> 예) A 가 (3 × 2) 이면 A · A 는 곱할 수 없지만 A<sup>T</sup> · A 는 (2 × 3) · (3 × 2) = ( 　 × 　 ) 로 계산 가능!

In [ ]:
A = np.array([[1, 2], [3, 4], [5, 6]])
print('A.shape =', A.shape, '→ A.T.shape =', A.T.shape)
print(A.T)
print('A.T · A =\n', np.dot(A.T, A))

## 📖 개념 다지기 6. 원하는 부분만 꺼내기 : 인덱싱과 슬라이싱

　A = ( 10　20 ) / ( 30　40 ) / ( 50　60 )  (3 × 2)

| 코드 | 뜻 | 결과 |
|:---|:---|:---|
| `A[0, 0]` | 1행 1열 (번호는 **0부터**) | 10 |
| `A[2, 1]` | 3행 2열 | ( 　 ) |
| `A[:, 0]` | 모든 행의 1열 | [10, 30, 50] |
| `A[0:-1, 1:2]` | 1행 ~ 2행, 2열 | [[20], [40]] |
| `A[:, -1]` | 모든 행의 **마지막** 열 | ( 　　　　 ) |

> 💡 **머신러닝에서**: 데이터 표에서 앞쪽 열은 입력 x, 마지막 열은 정답 t 로 나눌 때 `data[:, 0:-1]`, `data[:, [-1]]` 을 씁니다.

In [ ]:
A = np.array([10, 20, 30, 40, 50, 60]).reshape(3, 2)
print(A)
print('A[0, 0]      =', A[0, 0])
print('A[2, 1]      =', A[2, 1])
print('A[:, 0]      =', A[:, 0])
print('A[0:-1, 1:2] =', A[0:-1, 1:2].tolist())
print('A[:, -1]     =', A[:, -1])

## 💻 실습. 시험 점수 데이터를 입력과 정답으로 나누기

아래 표는 학생 5명의 1차, 2차, 3차 시험 점수(입력)와 최종 점수(정답) 입니다. (원본 강의의 data-01.csv 일부)

| 1차 | 2차 | 3차 | 최종 |
|:-:|:-:|:-:|:-:|
| 73 | 80 | 75 | 152 |
| 93 | 88 | 93 | 185 |
| 89 | 91 | 90 | 180 |
| 96 | 98 | 100 | 196 |
| 73 | 66 | 70 | 142 |

In [ ]:
# 파일 만들기 (실제로는 np.loadtxt 로 csv 파일을 읽어 옵니다)
with open('data-01.csv', 'w') as f:
    f.write('73,80,75,152\n93,88,93,185\n89,91,90,180\n96,98,100,196\n73,66,70,142\n')

loaded_data = np.loadtxt('./data-01.csv', delimiter=',', dtype=np.float32)
x_data = loaded_data[:, 0:-1]     # 입력: 처음부터 마지막 열 '전'까지
t_data = loaded_data[:, [-1]]     # 정답: 마지막 열 (2차원 모양 유지)

print('x_data.shape =', x_data.shape, ', t_data.shape =', t_data.shape)
print('입력 x_data =\n', x_data)
print('정답 t_data =\n', t_data)

> **[확인 2]** `x_data` 의 형상은 ( 　 × 　 ), `t_data` 의 형상은 ( 　 × 　 ) 이다.
> 가중치 W 를 곱해 x_data · W 의 형상이 t_data 와 같은 (5 × 1) 이 되려면, W 의 형상은 ( 　 × 　 ) 이어야 한다.

### 자주 쓰는 numpy 함수

In [ ]:
X = np.array([2, 4, 6, 8])
print('np.sum(X) =', np.sum(X), ', np.max(X) =', np.max(X), ', np.argmax(X) =', np.argmax(X))
print('np.exp(X) =', np.exp(X))
print('np.log(X) =', np.log(X))

M = np.array([[2, 9, 6], [7, 2, 3], [0, 5, 8]])
print('\nM =\n', M)
print('np.max(M, axis=0)    =', np.max(M, axis=0), '  ← 열마다 최댓값 (세로 방향)')
print('np.max(M, axis=1)    =', np.max(M, axis=1), '  ← 행마다 최댓값 (가로 방향)')
print('np.argmax(M, axis=1) =', np.argmax(M, axis=1), '  ← 행마다 최댓값의 위치')

> 💡 `argmax` 는 손글씨 인식에서 "10개의 출력 중 **가장 큰 값의 위치** = 인공지능이 고른 숫자" 를 찾을 때 씁니다.

## ✏️ 연습 문제

> **[문제 1]** P = ( 1　0 ) / ( 2　1 ),  Q = ( 3　1 ) / ( 1　2 ) 일 때 P · Q 를 손으로 구하세요.
> P · Q = ( 　　 　　 ) / ( 　　 　　 )

> **[문제 2]** 1부터 12까지의 수로 3 × 4 행렬 M 을 만들고, ① M 의 형상, ② M<sup>T</sup> 의 형상, ③ M 의 2행 전체, ④ M<sup>T</sup> · M 의 형상을 구하세요.

아래 셀의 `______` 를 채워 실행하세요.

In [ ]:
# [문제 1]
P = np.array([[1, 0], [2, 1]])
Q = np.array([[3, 1], [1, 2]])
print('P · Q =\n', ______)

# [문제 2]
M = np.arange(1, 13).reshape(______)
print('① M.shape   =', M.shape)
print('② M.T.shape =', ______)
print('③ M 의 2행  =', M[______])
print('④ (M.T · M).shape =', np.dot(______, ______).shape)

In [ ]:
#@title 🔑 정답 확인 (먼저 풀어 본 뒤에 실행하세요) { display-mode: "form" }
P = np.array([[1, 0], [2, 1]]); Q = np.array([[3, 1], [1, 2]])
print('문제 1 : P · Q =', np.dot(P, Q).tolist(), ' (1×3+0×1=3, 1×1+0×2=1, 2×3+1×1=7, 2×1+1×2=4)')
M = np.arange(1, 13).reshape(3, 4)
print('문제 2 : ① (3, 4)  ② (4, 3)  ③', M[1].tolist(), ' ④', np.dot(M.T, M).shape)

## 📌 정리

| 핵심 | 내용 |
|:---|:---|
| 형상 | `shape` 로 (행, 열) 확인. 계산 전에 항상 확인하는 습관! |
| 사칙연산 | 같은 자리끼리. 크기가 다르면 브로드캐스트 |
| 행렬 곱 | `np.dot(A, B)`, (m × n) · (n × k) = (m × k) |
| 전치 | `A.T`, 행 ↔ 열 |
| 슬라이싱 | `data[:, 0:-1]` 입력, `data[:, [-1]]` 정답 |

### 🏆 도전 과제
입력 데이터 X 의 형상이 (25 × 3) 이고 정답 T 의 형상이 (25 × 1) 일 때,
y = X · W + b 가 T 와 같은 형상이 되도록 하는 가중치 W 의 형상은 ( 　 × 　 ) 입니다.
→ 다음 시간(제 5편 선형회귀)에는 이 W 와 b 를 **인공지능이 스스로 찾는 방법** 을 배웁니다!

---

[⬅ 이전 : 제 2편 수치미분 - 파이썬으로 미분하기](https://colab.research.google.com/github/chenyenhee/ai-math-colab/blob/python/02_numerical_derivative.ipynb)　|　[📚 목차](https://colab.research.google.com/github/chenyenhee/ai-math-colab/blob/python/00_index.ipynb)　|　[다음 : 제 4편 머신러닝의 개념과 종류 ➡](https://colab.research.google.com/github/chenyenhee/ai-math-colab/blob/python/04_ml_overview.ipynb)